In [5]:
import architecture
import load_data
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_curve, auc

This Colab will contain the code you will need to convert your PyTorch Model into an ONNX network. In optimizing the model, we sacrifice usually a small amount of accuracy for a useful decrease in computation time. This code was taken from the following video; if you want to watch it here is the link:
https://www.youtube.com/watch?v=XdoHYgdIFrQ


Additional note: you may need to run:

python 3 -m pip install onnx onnxscript

In terminal if you get a ModuleNotFound error. Or, if using jupyter notebook, run:

import sys

!{sys.executable} -m pip install onnx onnxscript

In the notebook


In [6]:
##Variables to be defined##
img_height=256
img_width=256
device = 'cpu'

In [7]:
## Converting to ONNX ##
import sys

test_input=torch.randn(1,3,img_height,img_width, device=device) #not a real example, just used for tracing

#ONNX export
model = architecture.SimpleCNN()
model.load_state_dict(torch.load("finetuned.pth"))
model = model.to("cpu" \
"")
model.eval()
torch.onnx.export(
    model, #trained model
    test_input, #example template
    'onnx_exported_model.onnx', #save name
    opset_version=18, #Version of ONNX to use
    export_params=True, #storing weights
    do_constant_folding=True, #Using constant folding method for optimization
    input_names=['input'], #Define input names
    output_names=['output'], #define output names
)

[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/opt/anaconda3/envs/GSET/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.13.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,3,256,256]>
            ),
            outputs=(
                %"output"<FLOAT,[1,1]>
            ),
            initializers=(
                %"features.0.weight"<FLOAT,[32,3,3,3]>{TorchTensor(...)},
                %"features.0.bias"<FLOAT,[32]>{TorchTensor(...)},
                %"features.2.weight"<FLOAT,[32,32,3,3]>{TorchTensor(...)},
                %"features.2.bias"<FLOAT,[32]>{TorchTensor(...)},
                %"features.5.weight"<FLOAT,[64,32,3,3]>{TorchTensor(...)},
                %"features.5.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"features.7.weight"<FLOAT,[64,64,3,3]>{TorchTensor(...)},
                %"features.7.bias"

To confirm everything is working correctly, do:

pip install onnxruntime

in the terminal, or if jupyter:

!{sys.executable} -m pip install onnxruntime

and run the following code:


In [8]:
transform = transforms.Compose([transforms.Resize((256, 256)),transforms.ToTensor()])
test_dataset = load_data.Dataset(False, True, False, True, False, False, transform)
test_loader = DataLoader(test_dataset, 1, True)

In [9]:
import onnxruntime as ort

session = ort.InferenceSession("onnx_exported_model.onnx")

onnx_predictions = []
onnx_labels = []

input_name = session.get_inputs()[0].name

for images, labels in test_loader:

    # Loop through each image in the batch
    for i in range(images.size(0)):

        # Create a batch of size 1
        image = images[i:i+1].numpy()

        # ONNX inference
        output = session.run(None,{input_name: image})[0]

        # Sigmoid
        probability = 1 / (1 + np.exp(-output))

        # Binary prediction
        prediction = (probability >= 0.5).astype(np.int64)

        onnx_predictions.append(prediction.item())
        onnx_labels.append(labels[i].item())

# Evaluate
accuracy = accuracy_score(onnx_labels, onnx_predictions)
print(f"ONNX Accuracy: {accuracy*100:.2f}%")

print(classification_report(onnx_labels,onnx_predictions))

ONNX Accuracy: 89.13%
              precision    recall  f1-score   support

           0       0.85      0.95      0.89       112
           1       0.94      0.84      0.89       118

    accuracy                           0.89       230
   macro avg       0.90      0.89      0.89       230
weighted avg       0.90      0.89      0.89       230

